# Synthetic Technical Q&A Dataset Generator

Instead of just **asking** a Q&A tool questions,
we now **generate entire Q&A datasets** using both cloud and local HuggingFace models.

In [ ]:
import os
import re
import json
import torch
import pandas as pd
import gradio as gr
from io import StringIO
from openai import OpenAI
from dotenv import load_dotenv
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextStreamer,
    BitsAndBytesConfig,
)

load_dotenv(override=True)

openrouter_client = OpenAI(
    api_key=os.getenv('OPENROUTER_API_KEY'),
    base_url="https://openrouter.ai/api/v1",
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Topics & Prompt Design

In [ ]:
TOPICS = {
    "Week 1 — LLM APIs & Prompting": [
        "OpenAI API usage and parameters",
        "System vs user messages",
        "Prompt engineering techniques",
        "Streaming responses",
        "JSON mode and structured outputs",
        "Token counting and API pricing",
        "OpenRouter and Ollama",
    ],
    "Week 2 — Tool Calling & Agents": [
        "Function calling syntax and format",
        "Tool definitions and JSON schemas",
        "Agent patterns and agentic loops",
        "Parallel function calling",
        "Structured outputs with Pydantic",
        "Gradio ChatInterface",
    ],
    "Week 3 — Transformers & HuggingFace": [
        "AutoTokenizer and AutoModelForCausalLM",
        "BPE vs WordPiece tokenization",
        "HuggingFace pipelines",
        "4-bit quantization with BitsAndBytes",
        "apply_chat_template",
        "Encoder vs decoder architectures",
        "Whisper speech-to-text",
    ],
}

DIFFICULTY_LEVELS = ["Beginner", "Intermediate", "Advanced"]

SYSTEM_PROMPT = """\
You are an expert educator creating high-quality technical Q&A pairs for an LLM Engineering course.
Generate EXACTLY the number of Q&A pairs requested.

Output each pair in this EXACT format — nothing else:

Q: <question>
A: <answer>
---

Rules:
- One question, one concise answer per pair.
- Questions should be specific and unambiguous.
- Answers should be 1-3 sentences, technically accurate.
- Vary which concepts are covered across the pairs.
"""

def build_user_prompt(topic_key: str, difficulty: str, num_pairs: int) -> str:
    concepts = ", ".join(TOPICS[topic_key])
    return (
        f"Generate {num_pairs} {difficulty}-level Q&A pairs about: {topic_key}.\n"
        f"Draw from these concepts: {concepts}."
    )

print(f"Topics defined: {list(TOPICS.keys())}")

## HuggingFace Local Model: AutoTokenizer + AutoModelForCausalLM

We use **TinyLlama-1.1B-Chat** which is a compact model that runs on CPU.
4-bit quantization is applied automatically when a GPU + bitsandbytes are available.

In [ ]:
HF_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

_hf_tokenizer = None
_hf_model     = None

def load_hf_model():
    """Lazy-load TinyLlama once; reuse on subsequent calls."""
    global _hf_tokenizer, _hf_model
    if _hf_model is not None:
        return _hf_tokenizer, _hf_model

    print(f"Loading {HF_MODEL_NAME} …")
    _hf_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
    _hf_tokenizer.pad_token = _hf_tokenizer.eos_token

    # Use 4-bit quantization on GPU if bitsandbytes is available
    quant_config = None
    if device == "cuda":
        try:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_quant_type="nf4",
            )
            print("  4-bit quantization enabled (GPU)")
        except Exception:
            print("  bitsandbytes not available — loading in full precision")

    _hf_model = AutoModelForCausalLM.from_pretrained(
        HF_MODEL_NAME,
        quantization_config=quant_config,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    if device == "cpu":
        _hf_model = _hf_model.to("cpu")
    _hf_model.eval()
    print(f"  ✅ Model loaded on {device}")
    return _hf_tokenizer, _hf_model


def generate_hf(messages: list[dict], max_new_tokens: int = 600) -> str:
    """Run inference via AutoModelForCausalLM with apply_chat_template."""
    tokenizer, model = load_hf_model()

    # apply_chat_template wraps messages in the model's special tokens (Day 3)
    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device if device == "cuda" else "cpu")

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            streamer=streamer,
        )

    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("HuggingFace generator ready (model loaded lazily on first use).")

## Cloud Generator

In [ ]:
def generate_openrouter(messages: list[dict], model: str = "openai/gpt-4o-mini") -> str:
    """Generate via OpenRouter cloud API."""
    response = openrouter_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.8,
    )
    return response.choices[0].message.content.strip()


CLOUD_MODELS = {
    "GPT-4o-mini (OpenRouter)": "openai/gpt-4o-mini",
    "GPT-4.1-mini (OpenRouter)": "openai/gpt-4.1-mini",
    "Llama-3.1-8B (OpenRouter)": "meta-llama/llama-3.1-8b-instruct",
}

print("Cloud generator ready.")

## Parser: Raw LLM Text → Structured Q&A Records

In [ ]:
def parse_qa_pairs(raw: str, topic: str, difficulty: str) -> list[dict]:
    """Parse 'Q: ...\nA: ...\n---' blocks into structured dicts."""
    pairs  = []
    blocks = re.split(r'---+', raw)
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        q_match = re.search(r'Q:\s*(.+?)(?=\nA:|$)', block, re.DOTALL)
        a_match = re.search(r'A:\s*(.+)',              block, re.DOTALL)
        if q_match and a_match:
            pairs.append({
                "id":         len(pairs) + 1,
                "topic":      topic,
                "difficulty": difficulty,
                "question":   q_match.group(1).strip(),
                "answer":     a_match.group(1).strip(),
            })
    return pairs


def export_json(pairs: list[dict]) -> str:
    path = "qa_dataset.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(pairs, f, indent=2)
    return path


def export_csv(pairs: list[dict]) -> str:
    path = "qa_dataset.csv"
    pd.DataFrame(pairs).to_csv(path, index=False)
    return path


print("Parser and exporters ready.")

## Generation

In [ ]:
def generate_dataset(
    topic_key: str,
    difficulty: str,
    num_pairs: int,
    model_choice: str,
) -> tuple[list[dict], str]:
    """
    Generate synthetic Q&A pairs for the given topic and difficulty.
    Returns (pairs, raw_text).
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_prompt(topic_key, difficulty, num_pairs)},
    ]

    print(f"Generating {num_pairs} pairs | topic={topic_key} | difficulty={difficulty} | model={model_choice}")

    if model_choice == "TinyLlama (HF local)":
        raw = generate_hf(messages)
    else:
        raw = generate_openrouter(messages, model=CLOUD_MODELS[model_choice])

    pairs = parse_qa_pairs(raw, topic_key, difficulty)
    print(f"  Parsed {len(pairs)} / {num_pairs} Q&A pairs")
    return pairs, raw

## Smoke-Test

In [ ]:
sample_pairs, _ = generate_dataset(
    topic_key="Week 3 — Transformers & HuggingFace",
    difficulty="Intermediate",
    num_pairs=3,
    model_choice="GPT-4o-mini (OpenRouter)",
)

for p in sample_pairs:
    print(f"Q{p['id']}: {p['question']}")
    print(f"A{p['id']}: {p['answer']}")
    print()

## Gradio UI

Full UI with configuration, live preview, and JSON/CSV download.

In [ ]:
ALL_MODELS = list(CLOUD_MODELS.keys()) + ["TinyLlama (HF local)"]


def gradio_generate(topic_key, difficulty, num_pairs, model_choice, progress=gr.Progress()):
    progress(0, desc="Generating dataset…")

    try:
        pairs, raw = generate_dataset(topic_key, difficulty, int(num_pairs), model_choice)
    except Exception as exc:
        return f"**Error:** {exc}", None, None, gr.update(visible=False)

    if not pairs:
        return "No pairs parsed — try a different model or fewer pairs.", None, None, gr.update(visible=False)

    progress(0.8, desc="Exporting files…")
    json_path = export_json(pairs)
    csv_path  = export_csv(pairs)

    progress(1.0, desc="Done!")

    # Format preview markdown
    lines = [f"### ✅ Generated {len(pairs)} Q&A pairs\n"]
    for p in pairs:
        lines.append(f"**Q{p['id']}:** {p['question']}")
        lines.append(f"**A{p['id']}:** {p['answer']}")
        lines.append("---")
    preview = "\n\n".join(lines)

    return preview, json_path, csv_path, gr.update(visible=True)


with gr.Blocks(theme=gr.themes.Soft(), title="Synthetic Q&A Generator") as ui:
    gr.Markdown(
        "# 📚 Synthetic Technical Q&A Dataset Generator\n"
        "Generate labelled Q&A pairs using cloud (OpenRouter) or a local HuggingFace model, "
        "then download as JSON or CSV."
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Configuration")

            topic_dd = gr.Dropdown(
                choices=list(TOPICS.keys()),
                value=list(TOPICS.keys())[0],
                label="Topic",
            )
            difficulty_dd = gr.Dropdown(
                choices=DIFFICULTY_LEVELS,
                value="Intermediate",
                label="Difficulty",
            )
            num_slider = gr.Slider(
                minimum=3, maximum=20, step=1, value=5,
                label="Number of Q&A pairs",
            )
            model_dd = gr.Dropdown(
                choices=ALL_MODELS,
                value="GPT-4o-mini (OpenRouter)",
                label="Model",
                info="TinyLlama is downloaded on first use",
            )
            generate_btn = gr.Button("🚀 Generate Dataset", variant="primary", size="lg")

            gr.Markdown("### 📥 Download")
            with gr.Row():
                json_dl = gr.File(label="JSON",  interactive=False, visible=False)
                csv_dl  = gr.File(label="CSV",   interactive=False, visible=False)

        with gr.Column(scale=2):
            gr.Markdown("### 📝 Generated Q&A Pairs")
            preview_md = gr.Markdown(value="Click **Generate Dataset** to start…")

    def on_generate(topic, difficulty, num, model):
        preview, json_path, csv_path, dl_visibility = gradio_generate(
            topic, difficulty, num, model
        )
        json_update = gr.update(value=json_path, visible=(json_path is not None))
        csv_update  = gr.update(value=csv_path,  visible=(csv_path  is not None))
        return preview, json_update, csv_update

    generate_btn.click(
        fn=on_generate,
        inputs=[topic_dd, difficulty_dd, num_slider, model_dd],
        outputs=[preview_md, json_dl, csv_dl],
    )

ui.launch()